# Lesson 13 Walkthrough: Quantum Computing (a Bunny Trail)
**Artificial Intelligence — Medina County Career Center**

> **Setup (once):** this lesson uses **PennyLane**, a Python library for quantum computing. If it isn't installed, run this in a terminal (or a notebook cell):
> `pip install pennylane`

This is an **enrichment** side-trip: a first taste of a genuinely different kind of computing. The amazing part — you'll see at the end that we can run the **same gradient-descent training** you used on neural nets, but on a **quantum circuit**. Run each cell in order.

## Bits vs. Qubits

A normal **bit** is 0 or 1. A **qubit** can be 0, 1, or a blend of both at once — called **superposition** — until you **measure** it, at which point it randomly collapses to 0 or 1 with certain probabilities.

We describe a quantum program as a **circuit**: a `device` (the quantum computer, or a simulator) plus a `QNode` (the circuit function) built from **gates**.

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# a simulator with 2 qubits (called 'wires')
dev = qml.device('default.qubit', wires=2)

@qml.qnode(dev)
def superpose():
    qml.Hadamard(wires=0)      # the Hadamard gate puts a qubit into superposition
    return qml.probs(wires=0)  # probability of measuring 0 vs 1

probs = [round(float(p), 3) for p in superpose()]
print('measure-0 / measure-1 probabilities:', probs)   # Should print [0.5, 0.5]

A perfect 50/50 coin. The **Hadamard** gate is the classic way to create superposition. Now let's control the odds.

## Rotations & Expectation Values

The **RY(theta)** gate rotates a qubit by angle `theta`. We'll read it out with the **expectation value** of Z, which runs from **+1** (definitely 0) to **-1** (definitely 1), passing through **0** (a 50/50 mix).

In [ ]:
@qml.qnode(dev)
def rotate(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

print('RY(0)    <Z> =', round(float(rotate(0.0)), 3))       # Should print 1.0
print('RY(pi/2) <Z> =', round(float(rotate(np.pi/2)), 3))   # Should print 0.0
print('RY(pi)   <Z> =', round(float(rotate(np.pi)), 3))     # Should print -1.0

So `theta` steers the qubit from 0 (`<Z>=+1`) to 1 (`<Z>=-1`). Halfway (`pi/2`) is the 50/50 point. We can also read the raw probabilities:

In [ ]:
@qml.qnode(dev)
def rotate_probs(theta):
    qml.RY(theta, wires=0)
    return qml.probs(wires=0)

print('RY(pi/3) probs:', [round(float(p), 3) for p in rotate_probs(np.pi/3)])   # Should print [0.75, 0.25]

## Entanglement: the 'spooky' part

**Entanglement** links two qubits so that measuring one instantly tells you the other — even though each alone looks random. The famous **Bell state** uses a Hadamard plus a **CNOT** gate.

In [ ]:
@qml.qnode(dev)
def bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])       # entangle qubit 0 with qubit 1
    return qml.probs(wires=[0, 1])

# order is [00, 01, 10, 11]
print('Bell probs:', [round(float(p), 3) for p in bell()])   # Should print [0.5, 0.0, 0.0, 0.5]

Only **00** and **11** ever appear — never 01 or 10. The two qubits are perfectly correlated: if one is 0 the other is 0; if one is 1 the other is 1. That's entanglement, and it has no classical equivalent.

## The payoff: Quantum Machine Learning

Here's why PennyLane exists. A circuit with an adjustable angle is a **variational circuit** — basically a model with a **weight** (`theta`). PennyLane can compute the **gradient** of a quantum circuit, so we can **train** it with gradient descent — exactly like a neural network.

**Goal:** find the `theta` that flips |0> into |1> — i.e., drives `<Z>` down to -1. We *know* the answer is `pi`, but watch the optimizer discover it.

In [ ]:
@qml.qnode(dev)
def circuit(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

def cost(theta):
    return circuit(theta)          # minimizing <Z> pushes the qubit toward |1>

opt = qml.GradientDescentOptimizer(stepsize=0.3)
theta = np.array(0.1, requires_grad=True)
for step in range(60):
    theta = opt.step(cost, theta)

print('learned theta:', round(float(theta), 3))   # Should print 3.142
print('pi is:        ', round(float(np.pi), 3))
print('final <Z>:', round(float(circuit(theta)), 3))   # Should print -1.0

The optimizer climbed from `theta=0.1` to `theta=3.142` (that's **pi**) all by itself, using the same gradient-descent idea from your neural-network lesson — but the 'model' was a **quantum circuit**. That's the whole premise of quantum machine learning.

## Reality check
Today's real quantum computers are small and noisy (the 'NISQ' era). Nobody is replacing your laptop yet. But for a few special problems — chemistry/materials simulation, certain optimization and cryptography tasks — quantum machines could one day do things classical computers practically can't. Now you've written real quantum code. **On to the tasks!**